# Imports


In [21]:
import numpy as np
import torch
import time
import copy
import torch
import rendering
import heightfield
import image
import opt
import sys
import simannealing
from datetime import datetime

In [22]:
print("Running with " + str(sys.version))

Running with 3.8.11 (default, Aug  6 2021, 08:56:27) 
[Clang 10.0.0 ]


# Grad descent

In [31]:
def grad_descent(num_epochs, H, cam_directions, image_paths, use_barrier_loss = True, use_neighbor_loss = False, sim_annealing = True, optim_type = "Adam", heaviside_fn = None, heaviside_k = 0.1, num_repeats = 1, num_rays = 32):
  
  COL = heightfield.get_white_COL(H.shape[0]*num_repeats)
  now = datetime.now()
  date_time = now.strftime("%m_%d_%Y_%H-%M-%S")
  num_cams = len(cam_directions)
  num_strips = H.shape[0]*num_repeats
  ray_width = 1/num_rays
  
  original_ray_origins = [[], [], [], []]
  ray_origins = copy.deepcopy(original_ray_origins)
  coords = []
  
  schedule = torch.zeros(50) #update colors for 10 steps, then heights for 20 steps
  schedule[10:20] = 1
  schedule[40:50] = 1

  grad_types = schedule.repeat(20)
  

  for i in range(num_rays):
    for k in range(num_rays):
      coord = np.array([i*ray_width + ray_width/2, k*ray_width + ray_width/2, 1])
      coords.append(coord)

      for q in range(num_cams):   
        curr_origin = coord - 20*cam_directions[q]
        curr_origin[2] = curr_origin[2].item()
        ray_origins[q].append(curr_origin)



  H = torch.tensor(H, dtype=torch.float, requires_grad = True)
  COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)
  



  ray_packs = []


  for t in range(num_cams):
    ray_packs.append(rendering.get_ray_packs(ray_origins[t], cam_directions[t], num_strips))

  
  desired_cols = []
  for i in range(num_cams):
    desired_cols.append(image.get_tensor_from_image(image_paths[i], (64, 64), (num_rays, num_rays)))
  
  
  if sim_annealing:
    H, COL = simannealing.sim_annealing(H, COL, ray_packs, num_cams, num_repeats, heaviside_fn, heaviside_k, desired_cols)
      
  if optim_type == "Adam":
    optimizer = torch.optim.AdamW([H, COL],lr=0.1)
    
  if optim_type == "SGD":
    optimizer = torch.optim.SGD([H, COL],lr=0.1, momentum=0.9)


  for j in range(1, num_epochs):
    
    
    if j%20 == 0:
      
      repeated_H = H.repeat(num_repeats, num_repeats)
      torch.save(repeated_H, 'models/' + date_time + '_heights.pt')
      torch.save(COL, 'models/' + date_time + '_colors.pt')
      print("saving")
      
    change = False
    
    if j%100 == 0:
      H = heightfield.double_resolution(H)
      COL = heightfield.double_resolution(COL)
      H = torch.tensor(H, dtype=torch.float)
      COL = torch.tensor(COL, dtype=torch.float)
      num_strips = num_strips*2
      
      desired_cols = []
      for q in range(num_cams):

        desired_cols.append(image.get_tensor_from_image(image_paths[q], (64, 64), (num_rays, num_rays)))
      change = True


    if j%100 == 95:
      num_rays = num_rays*2
      ray_width = ray_width/2
      change = True
    
    if j%100 == 95 or j%100 == 0:
      coords = []
      change = True
      

      
      ray_origins = copy.deepcopy(original_ray_origins)
      for i in range(num_rays):
        for k in range(num_rays):
          coord = np.array([i*ray_width + ray_width/2, k*ray_width + ray_width/2, 1])
          coords.append(coord)
          for q in range(num_cams):    
            curr_origin = coord - 20*cam_directions[q] 
            curr_origin[2] = curr_origin[2].item()
            ray_origins[q].append(curr_origin)


      
      ray_packs = []
      for t in range(num_cams):
        ray_packs.append(rendering.get_ray_packs(ray_origins[t], cam_directions[t], num_strips))
        
      H.requires_grad = True
      COL.requires_grad = True
      
      desired_cols = []
      for i in range(num_cams):
        desired_cols.append(image.get_tensor_from_image(image_paths[i], (64, 64), (num_rays, num_rays)))

    if int(grad_types[j].item()) == 0:
      
      H.requires_grad = False
      COL.requires_grad = True
      

      lr = 0.05
      

    elif int(grad_types[j].item()) == 1:
      H.requires_grad = True
      COL.requires_grad = False 
      COL = COL.detach().numpy()
      COL = torch.tensor(COL, dtype=torch.float, requires_grad = False)
      if j > 49:
        lr = 0.03
      
    elif int(grad_types[j].item()) == 2:
      H.requires_grad = True
      COL.requires_grad = True
      H = torch.tensor(H, dtype=torch.float, requires_grad = True)
      COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)

    if j > 0:
      if int(grad_types[j].item()) != int(grad_types[j-1].item()):
        
        change = True
    if change:



      if optim_type == "Adam":
        optimizer = torch.optim.AdamW([H, COL],lr=lr)
      if optim_type == "SGD":
        optimizer = torch.optim.SGD([H, COL],lr=lr, momentum=0.9)




    result_cols = rendering.forward(H, COL, ray_packs, num_cams, num_repeats=num_repeats, heaviside_fn = heaviside_fn, heaviside_k = heaviside_k)

    mse_loss = 0
    for q in range(num_cams):
      mse_loss += torch.nn.functional.mse_loss(result_cols[q].double(), desired_cols[q].double())




    if j % 1 == 0 or j == 99:
      print("EPOCH: " + str(j) + "\t LOSS: " + str(mse_loss.item()/2))
      time.sleep(1)
    if use_barrier_loss:
      barrier_loss = opt.barrier_fn(H, min_val = 0.1, max_val = 1.5)
      mse_loss = mse_loss + 0.02*barrier_loss

    if use_neighbor_loss:
      neighbor_loss = opt.neighbor_fn(H)
      mse_loss = mse_loss + 0.5*neighbor_loss



      
       
    
    optimizer.zero_grad()
    mse_loss.backward()
    optimizer.step()
    




  repeated_H = H.repeat(num_repeats, num_repeats)


  return repeated_H, COL, result_cols, desired_cols



# Main

In [32]:
H = heightfield.get_uniform_H(4, 4, center=0.5)

cam_direction_1 = np.array([1, 0.000001, -1])

cos = [1, 0.94, 0.766, 0.5, 0.174, -0.174, -0.5, -0.766, -0.94, -1]
sin = [0.001, 0.34, 0.643, 0.866, 0.984, 0.984, 0.866, 0.643, 0.34, 0.001]

np.set_printoptions(threshold=1000)
torch.set_printoptions(threshold = 1000)

cam_direction_1 = np.array([1, 0.0001, -2])
cam_direction_2 = np.array([1, 0.0001, -10])
cam_direction_3 = np.array([-1, 0.00001, -2])

cam_directions = np.array([cam_direction_1, cam_direction_3])#, cam_direction_2, cam_direction_3])

num_epochs = 260
p1 = "test_images/forest.png"
p2 = "test_images/cat.png"
p3 = "test_images/clouds.png"
image_paths = [p1, p3]
H1, COL1, result_cols, desired_cols = grad_descent(num_epochs, H, cam_directions, image_paths, sim_annealing = False, use_barrier_loss = True, use_neighbor_loss = False, optim_type="Adam", heaviside_fn = "tan", num_rays = 16, num_repeats=2)




/var/folders/1y/91nvhzqn12d2sh6dm1wf4_lh0000gn/T/ipykernel_9626/1749727662.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  H = torch.tensor(H, dtype=torch.float, requires_grad = True)
/var/folders/1y/91nvhzqn12d2sh6dm1wf4_lh0000gn/T/ipykernel_9626/1749727662.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)


EPOCH: 1	 LOSS: 0.317644807105388
EPOCH: 2	 LOSS: 0.2291914204404103
EPOCH: 3	 LOSS: 0.16171591779354272
EPOCH: 4	 LOSS: 0.1135639447156021
EPOCH: 5	 LOSS: 0.0813026200074681
EPOCH: 6	 LOSS: 0.06128989696788263
EPOCH: 7	 LOSS: 0.05024286525120677
EPOCH: 8	 LOSS: 0.045292860549496466
EPOCH: 9	 LOSS: 0.04411329891024101
EPOCH: 10	 LOSS: 0.04494571829988238
EPOCH: 11	 LOSS: 0.04494571829988238
EPOCH: 12	 LOSS: 0.04903015687724062
EPOCH: 13	 LOSS: 0.046989182408492314
EPOCH: 14	 LOSS: 0.04800700726158147
EPOCH: 15	 LOSS: 0.04963407320922486
EPOCH: 16	 LOSS: 0.048867543605150526
EPOCH: 17	 LOSS: 0.049285771510513325
EPOCH: 18	 LOSS: 0.04995186074079709
EPOCH: 19	 LOSS: 0.05188666508754665
saving
EPOCH: 20	 LOSS: 0.05069176458601282
EPOCH: 21	 LOSS: 0.04201146160003409
EPOCH: 22	 LOSS: 0.0364588365580851
EPOCH: 23	 LOSS: 0.03357319254306589
EPOCH: 24	 LOSS: 0.032392304395896354
EPOCH: 25	 LOSS: 0.031970656546522175
EPOCH: 26	 LOSS: 0.03184644797448566
EPOCH: 27	 LOSS: 0.0317877291595422
EPOC

/var/folders/1y/91nvhzqn12d2sh6dm1wf4_lh0000gn/T/ipykernel_9626/1749727662.py:76: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  H = torch.tensor(H, dtype=torch.float)
/var/folders/1y/91nvhzqn12d2sh6dm1wf4_lh0000gn/T/ipykernel_9626/1749727662.py:77: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  COL = torch.tensor(COL, dtype=torch.float)


EPOCH: 100	 LOSS: 0.03462901160922547
EPOCH: 101	 LOSS: 0.030559270015339313
EPOCH: 102	 LOSS: 0.02835095434845342
EPOCH: 103	 LOSS: 0.027706913257042762
EPOCH: 104	 LOSS: 0.027681087562774515
EPOCH: 105	 LOSS: 0.02761753552161271
EPOCH: 106	 LOSS: 0.02750353350457651
EPOCH: 107	 LOSS: 0.027445566593903393
EPOCH: 108	 LOSS: 0.027421008262121432
EPOCH: 109	 LOSS: 0.027314855221514986
EPOCH: 110	 LOSS: 0.027114951975558264
EPOCH: 111	 LOSS: 0.028260648542107518
EPOCH: 112	 LOSS: 0.029347501601548878
EPOCH: 113	 LOSS: 0.031198734466726382
EPOCH: 114	 LOSS: 0.03126391183409672
EPOCH: 115	 LOSS: 0.03215635636410005
EPOCH: 116	 LOSS: 0.03403865865673878
EPOCH: 117	 LOSS: 0.03306627989295055
EPOCH: 118	 LOSS: 0.035420034317873064
EPOCH: 119	 LOSS: 0.0367737345430364
saving
EPOCH: 120	 LOSS: 0.03635611851831269
EPOCH: 121	 LOSS: 0.03179271667327858
EPOCH: 122	 LOSS: 0.02891739101818773
EPOCH: 123	 LOSS: 0.027719538578653433
EPOCH: 124	 LOSS: 0.027435662329776813
EPOCH: 125	 LOSS: 0.02736233898